# 01 · Datos — PanNuke 256 px (Breast, Colon, Lung) sin imágenes dañadas

**Qué hace este notebook**
1. Lee tu carpeta `.arrow` de PanNuke y se queda solo con **Breast, Colon y Lung**.
2. Decodifica cada imagen **dentro de un `try/except`** (una imagen corrupta ya no rompe todo) y mide su calidad.
3. Marca como dañadas las que cumplan algún criterio (`cfg.QUALITY` en `fg_config.py`) y te muestra **galerías para verificar a ojo**.
4. Guarda solo las buenas en Drive (`X256.npy`, `y.npy`, `meta.json`, `split.npz`) + `quality_report.csv`.

**Por qué 256 px sin reescalar:** PanNuke ya es 256×256 nativo. No hay interpolación ni pérdida de detalle.

**Criterios de "dañada"** (cada uno con su justificación)
| Razón | Cómo se detecta | Qué atrapa |
|---|---|---|
| `corrupta` | no decodifica | archivos rotos |
| `oscura` | brillo medio < 100 | tu hallazgo previo con Colon |
| `sin_contraste` | desviación del gris < 15 | imágenes planas / vacías |
| `zonas_negras` | > 10 % de píxeles casi negros | relleno o corrupción parcial |
| `mayormente_blanca` | > 60 % casi blancos | vidrio sin tejido |
| `casi_gris` | saturación media < 0.06 | sin tinción H&E |
| `tono_no_HE` | (R+B)/2 − G < 0 | colores no rosa/morado |
| `borrosa` | nitidez muy baja respecto a la mediana **de su órgano** | fuera de foco |
| `duplicada` | mismo MD5 que otra | imágenes repetidas (evita fugas train/val) |

> No conozco *cuáles* son las dañadas de tu copia: por eso los umbrales son **configurables** y hay galerías de revisión.

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/fractal_histopatologia_256/code")   # carpeta con fg_*.py

!pip -q install datasets scipy pandas scikit-learn tqdm
import io, json, re, shutil
import numpy as np, pandas as pd
import fg_config as cfg, fg_quality as fq
for d in (cfg.PROJECT_DIR, cfg.DATA_DIR, cfg.RES_DIR): os.makedirs(d, exist_ok=True)
print("Órganos:", cfg.ORGANS, "| tamaño:", cfg.IMG_SIZE, "| dataset:", cfg.DATASET_DIR)

## 1. Cargar el dataset y elegir órganos

In [ ]:
from datasets import load_from_disk, concatenate_datasets, Image as HFImage, ClassLabel, DatasetDict
from PIL import Image
from tqdm.auto import tqdm

def safe_name(s):
    s = re.sub(r"[\s/&]+", "_", str(s).strip())
    return re.sub(r"_+", "_", s).strip("_")

if not os.path.exists(cfg.LOCAL_ARROW):
    print("Copiando dataset a disco local (leer de Drive es lento)...")
    shutil.copytree(cfg.DATASET_DIR, cfg.LOCAL_ARROW)
ds = load_from_disk(cfg.LOCAL_ARROW)
if isinstance(ds, DatasetDict):
    ds = concatenate_datasets([ds[k] for k in ds.keys()])
print("Filas:", len(ds)); print(ds.features)

img_col = next(k for k, v in ds.features.items() if isinstance(v, HFImage))
cands = [k for k in ["tissue_name", "tissue", "tissue_type", "type", "organ", "label", "class"] if k in ds.features]
assert cands, f"No encuentro la columna del órgano. Columnas: {list(ds.features)}"
lab_col = cands[0]; feat = ds.features[lab_col]
names = np.array([safe_name(feat.int2str(v) if isinstance(feat, ClassLabel) else v) for v in ds[lab_col]])
uniq, cnt = np.unique(names, return_counts=True)
print(f"Imagen: '{img_col}' | órgano: '{lab_col}'")
print("Imágenes por órgano (dataset completo):", dict(zip(uniq, cnt)))

lower = {u.lower(): u for u in uniq}
missing = [o for o in cfg.ORGANS if safe_name(o).lower() not in lower]
assert not missing, f"Órganos no encontrados: {missing}. Válidos: {list(uniq)}"
chosen = [lower[safe_name(o).lower()] for o in cfg.ORGANS]      # el orden define el índice de clase
print("Órganos elegidos (orden = índice de clase):", chosen)

## 2. Decodificar + medir calidad (tarda unos minutos; solo la primera vez)

In [ ]:
idx = np.flatnonzero(np.isin(names, chosen))
sub = ds.select(idx.tolist()).select_columns([img_col]).cast_column(img_col, HFImage(decode=False))
N, S = len(idx), cfg.IMG_SIZE
X = np.zeros((N, S, S, 3), np.uint8)
rows, resized = [], 0
for k, ex in enumerate(tqdm(sub, total=N, desc="Decodificando + midiendo")):
    rec = dict(row_id=int(idx[k]), organ=names[idx[k]], decode_error=False, md5="", error="")
    try:
        d = ex[img_col]
        raw = d["bytes"] if d.get("bytes") is not None else open(d["path"], "rb").read()
        im = Image.open(io.BytesIO(raw)); im.load(); im = im.convert("RGB")
        if im.size != (S, S):
            im = im.resize((S, S), Image.LANCZOS); resized += 1
        a = np.asarray(im); X[k] = a
        rec.update(fq.image_metrics(a)); rec["md5"] = fq.md5_of(a)
    except Exception as e:
        rec["decode_error"] = True; rec["error"] = repr(e)[:80]
        rec.update({c: np.nan for c in fq.METRIC_COLS})
    rows.append(rec)
df_raw = pd.DataFrame(rows)
print(f"{N} imágenes | no decodificaron: {int(df_raw.decode_error.sum())} | reescaladas (no eran 256): {resized}")
df_raw.describe().round(2)

## 3. Marcar dañadas y revisar a ojo
Ejecuta esta sección todas las veces que quieras: si cambias `cfg.QUALITY` (o `cfg.MANUAL_BLACKLIST`) solo hay que **volver a correr las celdas de esta sección**, no hace falta decodificar de nuevo.

In [ ]:
df = fq.flag_quality(df_raw)
display(fq.summary(df))
display(fq.reasons_table(df))

In [ ]:
# Galerías de lo DESCARTADO, una por razón (¿de verdad están dañadas?)
t = fq.reasons_table(df)
for r in (t.index if len(t) else []):
    fq.show_rejected(X, df, r, n=16)

In [ ]:
# Galerías de lo CONSERVADO (¿se ve bien?) y de lo conservado más cercano al umbral (¿lo dejarías?)
fq.show_kept(X, df, n_per_organ=8)
fq.show_borderline(X, df, n=16)

In [ ]:
# Distribución de brillo por órgano, antes y después del filtro
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(chosen), figsize=(4 * len(chosen), 2.8), sharey=True)
for a, o in zip(np.atleast_1d(ax), chosen):
    d = df[df.organ == o]
    a.hist(d.mean_gray.dropna(), bins=40, alpha=.5, label="todas")
    a.hist(d[d.keep].mean_gray.dropna(), bins=40, alpha=.5, label="conservadas")
    a.set_title(o); a.set_xlabel("brillo medio")
np.atleast_1d(ax)[0].legend(); plt.tight_layout(); plt.show()

> **Si algo no te convence:** edita `QUALITY` en `fg_config.py` (o `cfg.QUALITY["min_mean"] = 90` en una celda) o añade `row_id` a `cfg.MANUAL_BLACKLIST`, y re-ejecuta las 4 celdas de arriba. Cuando las galerías te convenzan, sigue.

## 4. Guardar el dataset limpio

In [ ]:
from sklearn.model_selection import train_test_split
keep = df["keep"].to_numpy()
y_all = np.array([chosen.index(o) for o in df["organ"]])
Xc, yc = X[keep], y_all[keep]
tr, va = train_test_split(np.arange(len(Xc)), test_size=cfg.TEST_FRACTION, stratify=yc, random_state=cfg.SEED)

np.save(f"{cfg.DATA_DIR}/X256.npy", Xc)
np.save(f"{cfg.DATA_DIR}/y.npy", yc)
np.savez(f"{cfg.DATA_DIR}/split.npz", train=tr, val=va)
json.dump({"classes": chosen, "n_total": int(len(df)), "n_kept": int(keep.sum()),
           "kept_row_ids": df.loc[keep, "row_id"].astype(int).tolist(), "quality": cfg.QUALITY,
           "manual_blacklist": list(cfg.MANUAL_BLACKLIST)}, open(f"{cfg.DATA_DIR}/meta.json", "w"))
df.drop(columns=["md5"]).to_csv(f"{cfg.RES_DIR}/quality_report.csv", index=False)

print(f"Guardadas {len(Xc)} imágenes buenas de {len(df)}  ({100 * len(Xc) / len(df):.1f} %)")
print("Por órgano:", dict(zip(chosen, np.bincount(yc))))
print("Train:", np.bincount(yc[tr]), "| Val:", np.bincount(yc[va]))
print(f"X256.npy: {Xc.nbytes / 2**30:.2f} GB")

In [ ]:
# Verificación: recargar desde Drive y comprobar
Xv = np.load(f"{cfg.DATA_DIR}/X256.npy", mmap_mode="r"); yv = np.load(f"{cfg.DATA_DIR}/y.npy")
sp = np.load(f"{cfg.DATA_DIR}/split.npz")
assert Xv.shape[1:] == (256, 256, 3) and Xv.dtype == np.uint8 and len(Xv) == len(yv)
assert len(set(sp["train"]) & set(sp["val"])) == 0
print("OK:", Xv.shape, "| clases:", json.load(open(f"{cfg.DATA_DIR}/meta.json"))["classes"])